# Inventory Management Analysis
### Data Analytics Portfolio Project

**Tools:** Python, Pandas, NumPy, Matplotlib

**Dataset:** `sample_inventory_data.xlsx` — synthetic sample data created for portfolio demonstration. It does not represent any real company or customer.

## Project objective
Analyze inventory value, ageing, stock risk and ABC classification to identify items that may need management attention.

## Business questions
1. What is the total inventory quantity and value?
2. Which ageing groups hold the most inventory value?
3. Which parts have the highest inventory value?
4. Which items are old and high-value?
5. Which items fall into ABC categories A, B and C?
6. Which stock items should be prioritized for review?

> Note: Thresholds in this notebook are illustrative. A business should confirm its own policy before using them operationally.


## 1. Import libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:,.2f}".format)


## 2. Load the dataset
Keep `sample_inventory_data.xlsx` in the same folder as this notebook.

In [ ]:
from pathlib import Path

file_path = Path("sample_inventory_data.xlsx")

if not file_path.exists():
    raise FileNotFoundError(
        "sample_inventory_data.xlsx was not found. "
        "Place the Excel file in the same folder as this notebook."
    )

df = pd.read_excel(file_path)
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])
df.head()


## 3. Clean column names and inspect the data

In [ ]:
df.columns = (
    df.columns.astype(str)
    .str.strip()
    .str.lower()
    .str.replace(" ", "_", regex=False)
)

print(df.columns.tolist())
df.info()


In [ ]:
# Check missing values and duplicate records
missing_summary = df.isna().sum().sort_values(ascending=False)
print("Missing values by column:")
print(missing_summary)

print("\\nDuplicate rows:", df.duplicated().sum())

# Remove exact duplicate rows
df = df.drop_duplicates().copy()


## 4. Prepare data types

In [ ]:
df["buy_date"] = pd.to_datetime(df["buy_date"], errors="coerce")

numeric_columns = [
    "ageing_in_days", "unit_price",
    "inventory_quantity", "inventory_value"
]

for column in numeric_columns:
    df[column] = pd.to_numeric(df[column], errors="coerce")

# Keep rows with the core fields needed for the analysis
required_columns = ["part_number", "ageing_in_days", "unit_price",
                    "inventory_quantity", "inventory_value"]
df = df.dropna(subset=required_columns).copy()

print("Cleaned rows:", len(df))
df.head()


## 5. Data quality checks

In [ ]:
# Compare reported inventory value with quantity x unit price
df["calculated_value"] = (
    df["unit_price"] * df["inventory_quantity"]
).round(2)

df["value_difference"] = (
    df["inventory_value"] - df["calculated_value"]
).round(2)

print("Records with a value difference:",
      (df["value_difference"].abs() > 0.01).sum())

print("\\nZero or negative quantities:")
display(df[df["inventory_quantity"] <= 0][
    ["part_number", "inventory_quantity", "inventory_value"]
].head(10))

print("\\nZero or negative unit prices:")
display(df[df["unit_price"] <= 0][
    ["part_number", "unit_price"]
].head(10))


## 6. Basic inventory KPIs

In [ ]:
total_records = len(df)
unique_parts = df["part_number"].nunique()
total_quantity = df["inventory_quantity"].sum()
total_inventory_value = df["inventory_value"].sum()

print("Total inventory records:", total_records)
print("Unique part numbers:", unique_parts)
print("Total inventory quantity:", f"{total_quantity:,.0f}")
print("Total inventory value:", f"₹{total_inventory_value:,.2f}")


## 7. Inventory ageing analysis

In [ ]:
# Derive ageing groups consistently from the number of days
age_bins = [-1, 90, 180, 365, float("inf")]
age_labels = ["0-90 Days", "91-180 Days", "181-365 Days", "Above 365 Days"]

df["ageing_group"] = pd.cut(
    df["ageing_in_days"],
    bins=age_bins,
    labels=age_labels
)

ageing_analysis = (
    df.groupby("ageing_group", observed=False)["inventory_value"]
    .sum()
    .reindex(age_labels)
)

print(ageing_analysis)


In [ ]:
ax = ageing_analysis.plot(kind="bar", figsize=(9, 5))
ax.set_title("Inventory Value by Ageing Group")
ax.set_xlabel("Ageing Group")
ax.set_ylabel("Inventory Value (₹)")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()


## 8. Category-wise analysis

In [ ]:
category_analysis = (
    df.groupby("part_category", dropna=False)["inventory_value"]
    .sum()
    .sort_values(ascending=False)
)

print(category_analysis)

ax = category_analysis.plot(kind="bar", figsize=(10, 5))
ax.set_title("Inventory Value by Part Category")
ax.set_xlabel("Part Category")
ax.set_ylabel("Inventory Value (₹)")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()


## 9. Identify slow-moving and critical stock

In [ ]:
slow_stock = df[df["ageing_in_days"] > 180].copy()
critical_stock = df[df["ageing_in_days"] > 365].copy()

print("Stock older than 180 days — value:",
      f"₹{slow_stock['inventory_value'].sum():,.2f}")
print("Stock older than 365 days — value:",
      f"₹{critical_stock['inventory_value'].sum():,.2f}")

critical_stock.sort_values(
    "inventory_value", ascending=False
)[["part_number", "part_description", "ageing_in_days",
   "inventory_quantity", "inventory_value"]].head(15)


## 10. ABC analysis

In [ ]:
# Aggregate inventory value by part number
abc = (
    df.groupby("part_number", as_index=False)["inventory_value"]
    .sum()
    .sort_values("inventory_value", ascending=False)
)

grand_total = abc["inventory_value"].sum()

if grand_total > 0:
    abc["value_percentage"] = (
        abc["inventory_value"] / grand_total * 100
    )
else:
    abc["value_percentage"] = 0

abc["cumulative_percentage"] = abc["value_percentage"].cumsum()

# Assign category based on cumulative value contribution
abc["abc_category"] = np.select(
    [
        abc["cumulative_percentage"] <= 80,
        abc["cumulative_percentage"] <= 95
    ],
    ["A", "B"],
    default="C"
)

abc.head(15)


In [ ]:
abc_summary = (
    abc.groupby("abc_category")["inventory_value"]
    .sum()
    .reindex(["A", "B", "C"], fill_value=0)
)

print(abc_summary)

ax = abc_summary.plot(kind="bar", figsize=(7, 5))
ax.set_title("Inventory Value by ABC Category")
ax.set_xlabel("ABC Category")
ax.set_ylabel("Inventory Value (₹)")
plt.tight_layout()
plt.show()


## 11. Combine ABC category with ageing

In [ ]:
df = df.drop(columns=["abc_category"], errors="ignore")
df = df.merge(
    abc[["part_number", "abc_category"]],
    on="part_number",
    how="left"
)

priority_stock = df[
    (df["abc_category"] == "A") &
    (df["ageing_in_days"] > 365)
].copy()

print("Priority stock records:", len(priority_stock))
print("Priority stock value:",
      f"₹{priority_stock['inventory_value'].sum():,.2f}")

priority_stock.sort_values(
    "inventory_value", ascending=False
)[["part_number", "part_description", "ageing_in_days",
   "inventory_quantity", "inventory_value", "abc_category"]].head(20)


## 12. Inventory risk classification

In [ ]:
def classify_inventory_risk(row):
    if row["ageing_in_days"] > 365 and row["inventory_value"] > 10000:
        return "Very High Risk"
    elif row["ageing_in_days"] > 180 and row["inventory_value"] > 10000:
        return "High Risk"
    elif row["ageing_in_days"] > 180:
        return "Medium Risk"
    else:
        return "Low Risk"

df["inventory_risk"] = df.apply(classify_inventory_risk, axis=1)

risk_summary = (
    df.groupby("inventory_risk")["inventory_value"]
    .agg(["count", "sum"])
    .sort_values("sum", ascending=False)
)

risk_summary


## 13. Top 10 parts by inventory value

In [ ]:
top_parts = (
    df.groupby(["part_number", "part_description"], dropna=False)["inventory_value"]
    .sum()
    .sort_values(ascending=False)
    .head(10)
)

top_parts


In [ ]:
top_parts.sort_values().plot(kind="barh", figsize=(10, 6))
plt.title("Top 10 Parts by Inventory Value")
plt.xlabel("Inventory Value (₹)")
plt.ylabel("Part")
plt.tight_layout()
plt.show()


## 14. Final business summary

In [ ]:
over_180_value = df.loc[
    df["ageing_in_days"] > 180, "inventory_value"
].sum()

over_365_value = df.loc[
    df["ageing_in_days"] > 365, "inventory_value"
].sum()

priority_value = priority_stock["inventory_value"].sum()

print("========== INVENTORY ANALYSIS SUMMARY ==========")
print("Total records:", len(df))
print("Unique parts:", df["part_number"].nunique())
print("Total inventory quantity:", f"{df['inventory_quantity'].sum():,.0f}")
print("Total inventory value:", f"₹{df['inventory_value'].sum():,.2f}")
print("Value older than 180 days:", f"₹{over_180_value:,.2f}")
print("Value older than 365 days:", f"₹{over_365_value:,.2f}")
print("ABC-A stock older than 365 days:", f"₹{priority_value:,.2f}")


## 15. Business recommendations

Use the analysis to support decisions such as:

- Review high-value stock that has been ageing for more than 365 days.
- Investigate parts with zero or negative quantity or price.
- Review ABC-A items regularly because they contribute strongly to inventory value.
- Discuss replenishment and purchasing plans using actual consumption and lead-time data.
- Validate unusual inventory-value differences with the source records.

These are suggested review actions, not automatic conclusions. Confirm findings with the inventory and finance teams.

## 16. Limitations and next steps

- This sample dataset is synthetic and intended only to demonstrate the analysis workflow.
- Supplier performance, stock consumption, service usage and lead-time analysis require separate datasets. Those datasets are not included here.
- The risk thresholds and ABC cut-offs are illustrative and should be approved by the business.
- Before publishing a project based on real company data, obtain permission and remove confidential identifiers and commercially sensitive figures.


## 17. Export the analysis

In [ ]:
# Save the cleaned, analyzed sample data for further practice
output_path = Path("inventory_analysis_output.xlsx")
df.to_excel(output_path, index=False)
print("Saved:", output_path.resolve())
